# Phase 4 - train against the share of annotators who drew each pixel

One run, start to finish: train, score, tune the post-processing, predict the
test set, write a submission. Nothing has to come back to a workstation and
return, and nothing here holds a copy of the model code -- it is all imported
from the repository this notebook clones.

**What is being tested.** A frame three people annotated currently yields three
samples carrying three tracings that disagree by 2.6 filaments on average, and
the network is left to average them itself. The average it produces is not
calibrated: on fold 0 the probability map has a median of 0.006 and a 99th
percentile of 0.72, and every threshold from 0.3 to 0.7 scores within 0.001 of
the others, so the threshold is not a parameter at all. This run asks the same
samples for the *share* of annotators who drew each pixel instead, so that a
filament one of three people drew survives as 1/3 rather than being averaged
into something uncalibrated.

That matters because whether to emit such a filament is answerable. Emitting
one that k of n people drew, at IoU u, adds `k * u` to the numerator of
Panoptic Quality and `0.5 * n` to its denominator against staying silent, so it
pays when `k * u > 0.5 * n * PQ`. At PQ 0.40 a filament one of three drew is
worth emitting only if it can be drawn to IoU 0.6, while one two of three drew
is worth it at 0.3. A calibrated map turns that into a threshold.

**The number to beat is local PQ 0.3756** on fold 0's 142 validation frames,
which is what Phase 3 left. The comparison is made at Phase 3's own
post-processing settings, before anything is retuned, so that what moves is the
model and not the tuning.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Add the competition data as an input

Save with **Save Version**, and under *Advanced Settings* set **Save output**.
A Quick Save discards the output, which is where the checkpoint and the
probability maps go.

Expect about 75 minutes: 30 to train, 10 to write the validation and test
probability maps, 25 to sweep the post-processing, and the rest to write and
check the submission.

## 1. Clone the repository

Cloned rather than pip-installed: `configs/paths.yaml` and the frozen splits in
`configs/splits/` sit beside the package rather than inside it, and a wheel
would leave them behind -- the run would then be validated on a different set
of frames than every other run.

To reproduce this exact run later, put its commit hash in `REF`.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase4-vote-targets"  # or a commit hash, for a run to reproduce later
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; its CUDA build of torch stays as it is.
!pip install -q segmentation-models-pytorch

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
import torch

import filament

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print("filament", filament.__version__)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

`MAGFILO_ROOT` overrides the dataset root in `configs/paths.yaml`. The
annotation file is searched for rather than hard-coded, because the input
directory is named after whatever the competition data was attached as.

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

# <root>/train/<annotations> -> <root>
os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))
print("test images: ", len(list(paths.test_images.glob("*.jpeg"))))

WORK = Path("/kaggle/working")

## 3. Train

Fifteen epochs rather than forty: validation loss bottomed out at epoch 8 to 10
last time and the shorter schedule also scored better, 0.3649 against 0.3454.

`num_workers` is raised from the committed value, which is 0 for Windows where
every worker would re-read the 48 MB annotation file.

In [ ]:
from dataclasses import replace

from filament.training.config import TrainConfig
from filament.training.loop import train

config = replace(
    TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase4_votes.yaml"),
    num_workers=2,
    output_dir=WORK / "phase4_votes",
)
print(config)
assert config.vote_targets, "This notebook is the vote-share run; the config says otherwise."

In [ ]:
result = train(config)
print("best epoch", result.best_epoch, "validation loss", round(result.best_val_loss, 4))
print("checkpoint", result.checkpoint)

In [ ]:
epochs = [item.epoch for item in result.history]
figure, axes = plt.subplots(figsize=(7, 3.2))
axes.plot(
    epochs,
    [item.train_loss for item in result.history],
    label="train",
    linewidth=2,
    color="#2a78d6",
)
axes.plot(
    epochs,
    [item.val_loss for item in result.history],
    label="validation",
    linewidth=2,
    color="#eb6834",
)
axes.set_xlabel("epoch")
axes.set_ylabel("Dice + cross-entropy")
axes.set_title("Training and validation loss")
axes.legend(frameon=False)
axes.grid(axis="y", linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"training time: {sum(item.seconds for item in result.history) / 60:.1f} min")

## 4. Write the validation probability maps

Everything below reads these files rather than the model: thresholding, joining
fragments, filtering by area, scoring, and the sweep. Writing them once is what
keeps the rest off the GPU, and a byte per pixel resolves the probability far
finer than any threshold in use.

The solar disk is found here too. `extract_instances` needs it in the
coordinates of the map, and detecting it costs 20 ms a frame -- worth doing
once rather than inside every sweep point.

In [ ]:
from filament.data.coco import load_annotations
from filament.data.disk import detect_disk
from filament.data.image import load_grayscale
from filament.data.split import load_fold
from filament.evaluation import write_probability_maps
from filament.postprocess.search import load_maps
from filament.training.loop import load_checkpoint

model, stored = load_checkpoint(result.checkpoint)
dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(config.fold, f"{CHECKOUT}/configs/splits").val
print(f"fold {config.fold}: {len(val_stems)} validation frames")

VAL_MAPS = WORK / "prob_fold0_votes"
write_probability_maps(
    model,
    [paths.train_images / f"{stem}.jpeg" for stem in val_stems],
    VAL_MAPS,
    size=config.image_size,
    device="cuda",
)
val_maps = load_maps(VAL_MAPS)

val_disks = {}
for stem in val_stems:
    frame = load_grayscale(paths.train_images / f"{stem}.jpeg")
    val_disks[stem] = detect_disk(frame).scaled(config.image_size / frame.shape[0])

size_mb = sum(path.stat().st_size for path in VAL_MAPS.glob("*.npy")) / 1e6
print(f"{len(val_maps)} maps, {size_mb:.0f} MB")

## 5. Score it at the settings Phase 3 left

The comparison that decides whether this change worked. Same 142 frames, same
evaluator, and the same post-processing that produced **PQ 0.3756** -- nothing
retuned yet, so what moves is the model.

Encoding the ground truth costs about as long as everything else here and never
changes, so it is built once and reused by every sweep point below.

In [ ]:
from filament.metrics.pq import compute_pq
from filament.postprocess.search import Setting, predict_from_maps
from filament.submit.rle import masks_to_gt_df

BASELINE = {"pq": 0.3756, "sq": 0.6555, "rq": 0.5730, "tp": 968, "fp": 616, "fn": 827}
FROZEN = Setting(
    {"threshold": 0.5, "min_area": 400, "join_gap": 24.0, "join_angle": 50.0, "join_offset": 20.0}
)

gt_df = masks_to_gt_df(dataset, val_stems)
print(f"{len(gt_df)} annotated filaments over {len(val_stems)} frames")

symmetric = compute_pq(gt_df, predict_from_maps(val_maps, FROZEN, val_disks))
print(f"vote shares, Phase 3 settings: {symmetric}")
print(
    f"Phase 2 model, same settings:  PQ={BASELINE['pq']:.4f} SQ={BASELINE['sq']:.4f} "
    f"RQ={BASELINE['rq']:.4f} TP={BASELINE['tp']} FP={BASELINE['fp']} FN={BASELINE['fn']}"
)
print(
    f"difference: {symmetric.pq - BASELINE['pq']:+.4f} PQ, "
    f"{symmetric.sq - BASELINE['sq']:+.4f} SQ, {symmetric.rq - BASELINE['rq']:+.4f} RQ"
)

## 6. Did the map become a vote share?

The point of the change is that the output should rise with how many annotators
drew a pixel, because that is what makes the threshold a decision rather than an
arbitrary cut. It is not guaranteed: cross-entropy pulls the output towards the
share while Dice pulls it towards one, and on a target of two thirds the two
settle around 0.75.

Two numbers answer it, and the distance from the diagonal is not one of them:
the background fills the bottom bin with 99.6% of every frame and drags any
pixel-weighted average to nothing.

- **middle_share** -- how much of the marked area the model put somewhere other
  than the two extremes. The previous model: **0.207**.
- **middle_span** -- how far the observed share of annotators moves across that
  middle. The previous model: **0.139**, from 0.30 to 0.44, so a threshold
  placed anywhere in there was choosing between pixels people had agreed about
  equally.

Either one staying where it was means this run did not do what it set out to
do, whatever the PQ says.

In [ ]:
from filament.metrics.calibration import calibration

BASELINE_CALIBRATION = {"middle_share": 0.207, "middle_span": 0.139}

curve = calibration(val_maps, dataset.by_stem())
calibration_table = pd.DataFrame(curve.to_rows())
print(f"vote shares:   {curve}")
print(
    f"Phase 2 model: middle_share={BASELINE_CALIBRATION['middle_share']:.3f} "
    f"middle_span={BASELINE_CALIBRATION['middle_span']:.3f}"
)
calibration_table

In [ ]:
populated = curve.populated
figure, axes = plt.subplots(figsize=(4.6, 4.4))
axes.plot([0, 1], [0, 1], linewidth=1, color="#b8b5ad", linestyle="--", label="perfect")
axes.plot(
    [item.predicted for item in populated],
    [item.observed for item in populated],
    marker="o",
    linewidth=2,
    color="#2a78d6",
    label="measured",
)
axes.set_xlabel("probability the model gave")
axes.set_ylabel("share of annotators who drew it")
axes.set_title("Is the output a vote share?")
axes.set_xlim(0, 1)
axes.set_ylim(0, 1)
axes.legend(frameon=False)
axes.grid(linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(WORK / "calibration.png", dpi=140)
plt.show()

## 7. Sweep the post-processing

Two sweeps rather than one grid over everything: the threshold and the minimum
area interact, while the joining tolerances did not move the score at all last
time, so pairing them all would cost hundreds of points to learn the same
thing.

The threshold range is wider than Phase 3's. There it was swept from 0.3 to 0.7
and every value scored within 0.001; if the map now carries a share, the ends
should start to differ.

Each point is scored on all 142 frames. Splitting them to keep a half back was
considered and dropped: the plateau rule below already guards against fitting
the curve's accidents, the quantity a held-back half would measure came to
0.0016 in Phase 3, and halving the frames would make the curve itself noisier.

In [ ]:
from filament.postprocess.search import grid, sweep

thresholds = [round(0.1 * step, 2) for step in range(1, 10)]
first = sweep(
    val_maps,
    gt_df,
    grid(threshold=thresholds, min_area=[200, 400, 600], join_gap=[24.0]),
    disks=val_disks,
)
first.table

In [ ]:
threshold, threshold_plateau = first.plateau("threshold")
min_area, area_plateau = first.plateau("min_area")
print(f"threshold: plateau {threshold_plateau} -> {threshold}")
print(f"min_area:  plateau {area_plateau} -> {min_area}")
print(f"best single point: {first.best.setting} -> PQ {first.best.pq.pq:.4f}")

In [ ]:
second = sweep(
    val_maps,
    gt_df,
    grid(
        threshold=[threshold],
        min_area=[min_area],
        join_gap=[0.0, 8.0, 16.0, 24.0, 32.0, 48.0],
    ),
    disks=val_disks,
)
join_gap, gap_plateau = second.plateau("join_gap")
print(f"join_gap: plateau {gap_plateau} -> {join_gap}")
second.table

In [ ]:
CHOSEN = Setting(
    {
        "threshold": float(threshold),
        "min_area": int(min_area),
        "join_gap": float(join_gap),
        "join_angle": 50.0,
        "join_offset": 20.0,
    }
)
tuned = compute_pq(gt_df, predict_from_maps(val_maps, CHOSEN, val_disks))
print(f"chosen:    {CHOSEN}")
print(f"tuned:     {tuned}")
print(f"symmetric: {symmetric}")
print(f"Phase 3:   PQ={BASELINE['pq']:.4f}")

## 8. Predict the test set

The overlap check is not optional: Kaggle rejects a submission whose masks share
a pixel, and the rejected attempt still uses one of the five allowed per day.

Two submissions are written. The one to send is `submission.csv`, from the
settings the sweep chose. `submission_frozen.csv` uses Phase 3's settings
unchanged, as something to fall back on -- it costs one extra pass over maps
that are already in memory.

In [ ]:
from filament.metrics.overlap import check_no_overlap
from filament.submit.rle import write_submission

test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
TEST_MAPS = WORK / "prob_test_votes"
write_probability_maps(
    model,
    [paths.test_images / f"{stem}.jpeg" for stem in test_stems],
    TEST_MAPS,
    size=config.image_size,
    device="cuda",
)
test_maps = load_maps(TEST_MAPS)

test_disks = {}
for stem in test_stems:
    frame = load_grayscale(paths.test_images / f"{stem}.jpeg")
    test_disks[stem] = detect_disk(frame).scaled(config.image_size / frame.shape[0])
print(f"{len(test_maps)} test maps")

In [ ]:
submissions = {}
for name, setting in (("submission.csv", CHOSEN), ("submission_frozen.csv", FROZEN)):
    frame = predict_from_maps(test_maps, setting, test_disks)
    path = write_submission(frame, WORK / name)
    check_no_overlap(path)
    covered = frame["filament_id"].str.rsplit("_", n=1).str[0].nunique()
    submissions[name] = frame
    print(
        f"{name}: {len(frame)} masks over {covered} of {len(test_maps)} frames, "
        f"{len(test_maps) - covered} with none"
    )

submissions["submission.csv"].head()

## 9. Package what has to leave this machine

Only the small things. The test probability maps and the checkpoint stay in the
notebook output, where a later run can attach them as an input dataset rather
than anyone downloading and re-uploading them.

The validation maps do travel, because the analysis that decides what to try
next -- where the false positives come from, how far the masks fall short --
runs off them and should not need a GPU session each time.

Files are streamed into the archive from where they already are; nothing is
copied into a staging directory first.

In [ ]:
import json
import zipfile

summary = {
    "commit": REF,
    "symmetric": {
        "pq": symmetric.pq,
        "sq": symmetric.sq,
        "rq": symmetric.rq,
        "tp": symmetric.tp,
        "fp": symmetric.fp,
        "fn": symmetric.fn,
    },
    "tuned": {
        "pq": tuned.pq,
        "sq": tuned.sq,
        "rq": tuned.rq,
        "tp": tuned.tp,
        "fp": tuned.fp,
        "fn": tuned.fn,
    },
    "baseline": BASELINE,
    "chosen_settings": CHOSEN.values,
    "plateaus": {
        "threshold": threshold_plateau,
        "min_area": area_plateau,
        "join_gap": gap_plateau,
    },
    "calibration": {
        "monotonic": curve.is_monotonic,
        "middle_share": curve.middle_share,
        "middle_span": curve.middle_span,
        "spread": curve.spread,
        "mean_absolute_error": curve.mean_absolute_error,
        "baseline": BASELINE_CALIBRATION,
    },
    "best_epoch": result.best_epoch,
    "best_val_loss": result.best_val_loss,
    "training_minutes": sum(item.seconds for item in result.history) / 60,
}
(WORK / "phase4_summary.json").write_text(json.dumps(summary, indent=2))
first.table.to_csv(WORK / "sweep_threshold_area.csv", index=False)
second.table.to_csv(WORK / "sweep_join_gap.csv", index=False)
calibration_table.to_csv(WORK / "calibration.csv", index=False)

bundle = WORK / "phase4_votes_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for name in (
        "submission.csv",
        "submission_frozen.csv",
        "phase4_summary.json",
        "sweep_threshold_area.csv",
        "sweep_join_gap.csv",
        "calibration.csv",
        "calibration.png",
    ):
        archive.write(WORK / name, arcname=name)
    for path in sorted(VAL_MAPS.glob("*.npy")):
        archive.write(path, arcname=f"prob_fold0_votes/{path.name}")

print(f"{bundle.name}: {bundle.stat().st_size / 1e6:.0f} MB")
print(json.dumps(summary, indent=2))

## 10. What to report back

- The summary printed above, in full.
- Both sweep tables, so the plateau the notebook picked can be checked by hand.
- The calibration figure.

The decision to adopt is made on the **symmetric** comparison -- the same
post-processing as Phase 3 -- and needs `+0.01` PQ or better, plus an
explanation of what moved it. The tuned number is what the submission is worth,
not what the change is worth.

If the symmetric comparison is flat but the tuned one is well ahead, that is a
result too, and a different one: the model did not improve, but the map became
something the post-processing can work with.